## Load the Last.fm Dataset

In [26]:
from implicit.datasets.lastfm import get_lastfm

artists, users, artist_user_plays = get_lastfm()

In [24]:
print(f"Artists: {len(artists):,}")
print(f"User: {len(users):,}")
print(f"Matrix shape (artists x users): {artist_user_plays.shape}")
print(f"Non-zero interactions: {artist_user_plays.nnz:,}")

Artists: 292,385
User: 358,868
Matrix shape (artists x users): (292385, 358868)
Non-zero interactions: 17,535,606


## Preprocess Data for Training

- **`bm25_weight`**: Applies BM25 scoring to user-item interactions, reducing the influence of popular items.

- **`csr_matrix`**: A sparse matrix format (Compressed Sparse Row) that stores only non-zero values efficiently.

```python
# Dense matrix (wastes space)
dense = [
    [0, 0, 5],
    [3, 0, 0],
    [0, 2, 0]
]

# CSR format (stores only non-zero)
from scipy.sparse import csr_matrix
sparse = csr_matrix(dense)

print(sparse.data)    # [5, 3, 2] - values
print(sparse.indices) # [2, 0, 1] - columns  
print(sparse.indptr)  # [0, 1, 2, 3] - row starts

In [35]:
from implicit.nearest_neighbours import bm25_weight
from scipy.sparse import csr_matrix

# Apply BM25 weighting
weighted_plays = bm25_weight(X=artist_user_plays,  # Sparse matrix (artists x users) with play counts
                             K1=100,               # Saturation: higher = more weight for repeated plays
                             B=0.8                 # Normalization: higher = penalizes very active users more)
                             )    

# Transpose to (users x artists) format | rows -> users | columns -> items
user_plays = weighted_plays.T.tocsr()

print(f"Training matrix shape (users x artists): {user_plays.shape}")  

Training matrix shape (users x artists): (358868, 292385)


## Train ALS Model

**Confidence Weight**  
`Confidence = 1 + α × (interaction strength)`

Higher confidence = more important to the model.  
Common `α` values: 1.0 to 40.0.

| Platform | Action | Weight |
|----------|--------|--------|
| E-commerce | Purchase | 10.0 |
| E-commerce | Add to cart | 5.0 |
| E-commerce | Click | 1.0 |
| E-commerce | View | 0.5 |
| YouTube | Watch time (minutes) | 1.0 - 5.0 (scaled) |
| YouTube | Like | 10.0 |
| YouTube | Subscribe | 20.0 |

In [37]:
from implicit.als import AlternatingLeastSquares

# Initialize ALS model
als_model = AlternatingLeastSquares(factors=64,           # Latent factor dimenstion (embedding size)
                                    regularization=0.05,  # Prevents overfitting
                                    alpha=2.0,            # confidence weight for positive interactions (plays/listens)
                                    iterations=15)        # Training epochs

# Train the model
als_model.fit(user_items=user_plays,
              show_progress=True)

c:\Users\pouya\AppData\Local\Programs\Python\Python39\lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 15/15 [00:25<00:00,  1.73s/it]


## Examine Learned Factors


**What the numbers mean:**

- Each user has **64 numbers** representing their "taste vector"
- Each artist has **64 numbers** representing their "vibe vector"
- **Similar taste vectors × similar vibe vectors = high recommendation score**

In [38]:
print(f"User factors shape: {als_model.user_factors.shape}")  # (n_users, factors)
print(f"Item factors shape: {als_model.item_factors.shape}")  # (n_items, factors)

User factors shape: (358868, 64)
Item factors shape: (292385, 64)


In [50]:
# Embedding vector for first user
als_model.user_factors[1]

array([-3.2610247 , -6.745769  , -0.40691897, -3.2644763 ,  2.339559  ,
        0.55793524, -0.84855086,  2.9568052 ,  0.18997005, -2.281926  ,
        2.7796319 ,  0.9517909 , -1.5623076 , -4.161418  , -1.8845154 ,
        0.47798902, -1.4520152 ,  5.3343153 ,  2.8242505 , -4.5232587 ,
        2.0217876 , -3.2535677 , -0.7147659 , -2.315131  ,  1.7976915 ,
        2.7734625 , -3.334512  ,  3.5656066 , -3.179136  , -0.10863315,
        1.8583045 ,  0.12120786,  1.6612825 ,  0.74937326,  2.1102903 ,
       -0.23056032, -2.989192  , -1.8096565 , -0.7221807 , -1.1766083 ,
       -2.9434538 , -0.5917279 , -1.4647646 , -0.3938221 ,  4.8168526 ,
        0.73095286,  9.987825  , -1.0967163 ,  2.1833565 , -2.1423547 ,
       -1.0380671 ,  3.7168705 ,  5.0857944 , -3.6484063 ,  6.7515235 ,
        4.9169507 , -5.59382   ,  1.1816231 ,  0.9041562 ,  5.4210906 ,
       -5.8060575 , -2.5913982 ,  4.3389783 ,  2.5162072 ], dtype=float32)

## Make Recommendations

In [ ]:
user_id = 0

